In [1]:
import polars as pl
import numpy as np
import pandas as pd
import os
import pyarrow.parquet as pq
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

In [36]:
SEED = 42
LOAD_BATCH_SIZE = 75_000
SUBSAMPLE_SIZE = 170_000
TOP_K_FEATURES = 350  # Количество признаков для каждого таргета
SELECTED_FEATURES_PATH = "data/selected_features"
os.makedirs(SELECTED_FEATURES_PATH, exist_ok=True)

In [15]:
# ------------------------------
# 1. Объединение main и extra признаков
# ------------------------------
def build_wide_data():
    """Загружает и объединяет main и extra признаки"""
    print("Загрузка и объединение признаков...")
    
    train_customer_ids = pl.read_parquet('data/train_main_features.parquet').select('customer_id')
    test_customer_ids = pl.read_parquet('data/test_main_features.parquet').select('customer_id')
    
    # Используем ленивую загрузку для экономии памяти
    lf_main_train = pl.scan_parquet('data/train_main_features.parquet').drop('customer_id')
    lf_extra_train = pl.scan_parquet('data/train_extra_features.parquet').drop('customer_id')
    lf_main_test = pl.scan_parquet('data/test_main_features.parquet').drop('customer_id')
    lf_extra_test = pl.scan_parquet('data/test_extra_features.parquet').drop('customer_id')
    
    train_wide = pl.concat([lf_main_train, lf_extra_train], how='horizontal').with_columns(
        pl.all().cast(pl.Float16)
    ).collect()
    
    test_wide = pl.concat([lf_main_test, lf_extra_test], how='horizontal').with_columns(
        pl.all().cast(pl.Float16)
    ).collect()
    
    print(f"Train wide shape: {train_wide.shape}")
    print(f"Test wide shape: {test_wide.shape}")
    return train_wide, test_wide, train_customer_ids, test_customer_ids

# Сохраняем wide features для последующего использования
if not os.path.exists('data/train_wide_features.parquet'):
    print("Создаем train_wide_features.parquet...")
    train_wide, test_wide, train_customer_ids, test_customer_ids = build_wide_data()
    train_wide.write_parquet('data/train_wide_features.parquet')
    test_wide.write_parquet('data/test_wide_features.parquet')
    print("Файлы сохранены")
else:
    print("Загружаем существующие wide features...")
    train_wide = pl.read_parquet('data/train_wide_features.parquet')
    test_wide = pl.read_parquet('data/test_wide_features.parquet')
    train_customer_ids = pl.read_parquet('data/train_main_features.parquet').select('customer_id')
    test_customer_ids = pl.read_parquet('data/test_main_features.parquet').select('customer_id')

# Удаляем дубликаты
def remove_duplicate_columns(df):
    unique_cols = []
    seen = set()
    for col in df.columns:
        if col not in seen:
            unique_cols.append(col)
            seen.add(col)
    return df.select(unique_cols)

train_wide = remove_duplicate_columns(train_wide)
test_wide = remove_duplicate_columns(test_wide)
print(f"После удаления дубликатов: train shape {train_wide.shape}, test shape {test_wide.shape}")

Загружаем существующие wide features...
После удаления дубликатов: train shape (750000, 2440), test shape (250000, 2440)


In [35]:
# ------------------------------
# 2. Функции для отбора признаков
# ------------------------------
def load_parquet_subsample(path: str, idx: np.ndarray) -> pl.DataFrame:
    """Загружает подвыборку строк из parquet файла"""
    parts = []
    offset = 0
    
    for batch in pq.ParquetFile(path).iter_batches(batch_size=LOAD_BATCH_SIZE):
        left = np.searchsorted(idx, offset)
        right = np.searchsorted(idx, offset + batch.num_rows)
        if left < right:
            df_part = pl.from_arrow(batch)[(idx[left:right] - offset).tolist()]
            parts.append(df_part)
        offset += batch.num_rows
    
    result = pl.concat(parts, how="vertical")
    
    # Преобразуем категориальные признаки в int32
    cat_features = [col for col in result.columns if col.startswith("cat_feature")]
    for col in cat_features:
        result = result.with_columns(pl.col(col).cast(pl.Int32).fill_null(-1))
    
    return result

def subsample(target_size: int, y: np.ndarray) -> np.ndarray:
    """Создаёт сбалансированную подвыборку"""
    rng = np.random.default_rng(SEED)
    all_idx = np.arange(y.size)
    pos_idx = np.flatnonzero(y == 1)
    
    if pos_idx.size < target_size / 2:
        neg_idx = np.flatnonzero(y == 0)
        neg_take = rng.choice(neg_idx, size=target_size - pos_idx.size, replace=False)
        idx = np.concatenate((pos_idx, neg_take))
    else:
        idx = rng.choice(all_idx, size=target_size, replace=False)
    
    return np.sort(idx)

def select_topk_features_catboost(X: pl.DataFrame, y: np.ndarray, cat_features: list) -> list[str]:
    """Отбирает TOP_K_FEATURES наиболее важных признаков с помощью CatBoost"""
    feature_names = X.columns
    
    # Преобразуем Polars DataFrame в pandas и конвертируем категориальные признаки в int
    X_pd = X.to_pandas()
    
    # Конвертируем категориальные признаки в целые числа
    for col in cat_features:
        if col in X_pd.columns:
            # Заменяем NaN на -1 и конвертируем в int
            X_pd[col] = X_pd[col].fillna(-1).astype(int)
    
    # Разделяем на train/val
    train_idx, valid_idx = train_test_split(
        np.arange(len(y)), test_size=0.2, random_state=SEED, stratify=y
    )
    
    # Создаем Pool
    train_pool = Pool(
        X_pd.iloc[train_idx],
        label=y[train_idx],
        cat_features=cat_features
    )
    valid_pool = Pool(
        X_pd.iloc[valid_idx],
        label=y[valid_idx],
        cat_features=cat_features
    )
    
    # Обучаем CatBoost для оценки важности
    model = CatBoostClassifier(
        task_type="GPU",
        devices='0',      
        iterations=100,
        depth=5,
        metric_period=50, 
        learning_rate=0.1,
        loss_function='Logloss',
        eval_metric='AUC',
        early_stopping_rounds=20,
        verbose=False,
        random_seed=SEED
    )
    
    model.fit(train_pool, eval_set=valid_pool)
    
    # Получаем важность признаков
    feature_importance = model.get_feature_importance()
    feature_importance_dict = dict(zip(feature_names, feature_importance))
    
    # Сортируем и возвращаем топ признаков
    sorted_features = sorted(feature_names, key=lambda f: feature_importance_dict.get(f, 0.0), reverse=True)
    return sorted_features[:TOP_K_FEATURES]

def save_selected_features(path: str, features: list[str]) -> None:
    """Сохраняет список выбранных признаков в файл"""
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(features))

def load_selected_features(target_name: str) -> list[str]:
    """Загружает выбранные признаки для таргета"""
    feature_path = f"{SELECTED_FEATURES_PATH}/{target_name}.txt"
    if os.path.exists(feature_path):
        with open(feature_path, "r", encoding="utf-8") as f:
            return [line.strip() for line in f.readlines()]
    return None

In [27]:
def select_features_for_all_targets():
    """Выполняет отбор признаков для всех таргетов"""
    print("\n" + "="*60)
    print("ЭТАП 1: ОТБОР ПРИЗНАКОВ ДЛЯ КАЖДОГО ТАРГЕТА")
    print("="*60)
    
    target = pl.read_parquet('data/train_target.parquet')
    target_cols = [col for col in target.columns if col.startswith("target")]
    
    # Определяем категориальные признаки
    cat_features = [col for col in train_wide.columns if col.startswith("cat_feature")]
    
    for target_name in target_cols:
        print(f"\n----- Отбор признаков для {target_name} -----")
        feature_path = f"{SELECTED_FEATURES_PATH}/{target_name}.txt"
        
        if os.path.exists(feature_path):
            print(f"✓ Файл {feature_path} уже существует, пропускаем")
            continue
        
        # Загружаем целевые значения
        y = pl.read_parquet('data/train_target.parquet', columns=[target_name])[target_name].to_numpy()
        print(f"  Положительных: {int(y.sum())}, всего: {len(y)}")
        
        # Создаем сбалансированную подвыборку
        idx = subsample(SUBSAMPLE_SIZE, y)
        print(f"  Размер подвыборки: {len(idx)}")
        
        # Загружаем признаки для подвыборки
        X = load_parquet_subsample('data/train_wide_features.parquet', idx)
        print(f"  Загружено признаков: {X.shape[1]}")
        
        # Отбираем топ признаков
        try:
            selected_features = select_topk_features_catboost(X, y[idx], cat_features)
            print(f"  Выбрано {len(selected_features)} признаков")
            
            # Сохраняем
            save_selected_features(feature_path, selected_features)
            print(f"  ✓ Сохранено в {feature_path}")
        except Exception as e:
            print(f"  ❌ Ошибка при отборе признаков для {target_name}: {e}")
            # Сохраняем пустой список, чтобы не пытаться обработать позже
            save_selected_features(feature_path, [])

In [28]:
# ------------------------------
# 4. ЭТАП 2: Обучение моделей с использованием отобранных признаков
# ------------------------------
# Загружаем целевые переменные
target = pl.read_parquet('data/train_target.parquet')
target_cols = [col for col in target.columns if col.startswith("target")]
y_all = target.select(target_cols).to_pandas()

# Определяем категориальные признаки
cat_features = [col for col in train_wide.columns if col.startswith("cat_feature")]
print(f"\nКатегориальных признаков: {len(cat_features)}")

# Заполнение пропусков
def fill_missing(df, cat_features):
    df = df.clone()
    num_cols = [col for col in df.columns if col not in cat_features]
    
    for col in num_cols:
        median_val = df[col].drop_nulls().median()
        if median_val is not None and not np.isnan(median_val):
            df = df.with_columns(pl.col(col).fill_null(median_val))
        else:
            df = df.with_columns(pl.col(col).fill_null(0))
    
    for col in cat_features:
        df = df.with_columns(pl.col(col).fill_null(-1).cast(pl.Int32))
    
    return df

print("\nЗаполнение пропусков...")
train_wide = fill_missing(train_wide, cat_features)
test_wide = fill_missing(test_wide, cat_features)

# Преобразуем в pandas
X = train_wide.to_pandas()
X_test = test_wide.to_pandas()

# Разделяем на train/val
X_train, X_val, y_train, y_val = train_test_split(
    X, y_all, test_size=0.2, random_state=SEED,
    stratify=y_all['target_1_1']
)


Категориальных признаков: 67

Заполнение пропусков...


In [29]:
# ------------------------------
# 5. Разбиение на группы
# ------------------------------
num_groups = 4
group_size = len(target_cols) // num_groups
groups = [target_cols[i*group_size:(i+1)*group_size] for i in range(num_groups)]
# Добавляем остаток в последнюю группу
if len(groups[-1]) < len(target_cols) - (num_groups-1)*group_size:
    groups[-1].extend(target_cols[num_groups*group_size:])
print("Группы целевых переменных:")
for i, g in enumerate(groups):
    print(f"  Группа {i+1}: {len(g)} классов")

Группы целевых переменных:
  Группа 1: 10 классов
  Группа 2: 10 классов
  Группа 3: 10 классов
  Группа 4: 11 классов


In [38]:
best_params = {'target_1_1': {'iterations': 200,
  'learning_rate': 0.09237421878009823,
  'depth': 6,
  'l2_leaf_reg': 5,
  'random_strength': 1,
  'bagging_temperature': 0.6091961642353418,
  'border_count': 32,
  'min_data_in_leaf': 26},
 'target_1_2': {'iterations': 250,
  'learning_rate': 0.07076922519051033,
  'depth': 3,
  'l2_leaf_reg': 4,
  'random_strength': 3,
  'bagging_temperature': 0.5325152889039984,
  'border_count': 96,
  'min_data_in_leaf': 6},
 'target_1_3': {'iterations': 200,
  'learning_rate': 0.09237421878009823,
  'depth': 6,
  'l2_leaf_reg': 5,
  'random_strength': 1,
  'bagging_temperature': 0.6091961642353418,
  'border_count': 32,
  'min_data_in_leaf': 26},
 'target_1_4': {'iterations': 200,
  'learning_rate': 0.09237421878009823,
  'depth': 6,
  'l2_leaf_reg': 5,
  'random_strength': 1,
  'bagging_temperature': 0.6091961642353418,
  'border_count': 32,
  'min_data_in_leaf': 26},
 'target_1_5': {'iterations': 200,
  'learning_rate': 0.09237421878009823,
  'depth': 6,
  'l2_leaf_reg': 5,
  'random_strength': 1,
  'bagging_temperature': 0.6091961642353418,
  'border_count': 32,
  'min_data_in_leaf': 26},
 'target_2_1': {'iterations': 200,
  'learning_rate': 0.09237421878009823,
  'depth': 6,
  'l2_leaf_reg': 5,
  'random_strength': 1,
  'bagging_temperature': 0.6091961642353418,
  'border_count': 32,
  'min_data_in_leaf': 26},
 'target_2_2': {'iterations': 200,
  'learning_rate': 0.09237421878009823,
  'depth': 6,
  'l2_leaf_reg': 5,
  'random_strength': 1,
  'bagging_temperature': 0.6091961642353418,
  'border_count': 32,
  'min_data_in_leaf': 26},
 'target_2_3': {'iterations': 200,
  'learning_rate': 0.09237421878009823,
  'depth': 6,
  'l2_leaf_reg': 5,
  'random_strength': 1,
  'bagging_temperature': 0.6091961642353418,
  'border_count': 32,
  'min_data_in_leaf': 26},
 'target_2_4': {'iterations': 200,
  'learning_rate': 0.09237421878009823,
  'depth': 6,
  'l2_leaf_reg': 5,
  'random_strength': 1,
  'bagging_temperature': 0.6091961642353418,
  'border_count': 32,
  'min_data_in_leaf': 26},
 'target_2_5': {'iterations': 100,
  'learning_rate': 0.09210273435223537,
  'depth': 6,
  'l2_leaf_reg': 6,
  'random_strength': 2,
  'bagging_temperature': 0.5683704798044688,
  'border_count': 96,
  'min_data_in_leaf': 14},
 'target_2_6': {'iterations': 250,
  'learning_rate': 0.07076922519051033,
  'depth': 3,
  'l2_leaf_reg': 4,
  'random_strength': 3,
  'bagging_temperature': 0.5325152889039984,
  'border_count': 96,
  'min_data_in_leaf': 6},
 'target_2_7': {'iterations': 100,
  'learning_rate': 0.09210273435223537,
  'depth': 6,
  'l2_leaf_reg': 6,
  'random_strength': 2,
  'bagging_temperature': 0.5683704798044688,
  'border_count': 96,
  'min_data_in_leaf': 14},
 'target_2_8': {'iterations': 200,
  'learning_rate': 0.09237421878009823,
  'depth': 6,
  'l2_leaf_reg': 5,
  'random_strength': 1,
  'bagging_temperature': 0.6091961642353418,
  'border_count': 32,
  'min_data_in_leaf': 26},
 'target_3_1': {'iterations': 200,
  'learning_rate': 0.09237421878009823,
  'depth': 6,
  'l2_leaf_reg': 5,
  'random_strength': 1,
  'bagging_temperature': 0.6091961642353418,
  'border_count': 32,
  'min_data_in_leaf': 26},
 'target_3_2': {'iterations': 200,
  'learning_rate': 0.09237421878009823,
  'depth': 6,
  'l2_leaf_reg': 5,
  'random_strength': 1,
  'bagging_temperature': 0.6091961642353418,
  'border_count': 32,
  'min_data_in_leaf': 26},
 'target_3_3': {'iterations': 200,
  'learning_rate': 0.09237421878009823,
  'depth': 6,
  'l2_leaf_reg': 5,
  'random_strength': 1,
  'bagging_temperature': 0.6091961642353418,
  'border_count': 32,
  'min_data_in_leaf': 26},
 'target_3_4': {'iterations': 250,
  'learning_rate': 0.07076922519051033,
  'depth': 3,
  'l2_leaf_reg': 4,
  'random_strength': 3,
  'bagging_temperature': 0.5325152889039984,
  'border_count': 96,
  'min_data_in_leaf': 6},
 'target_3_5': {'iterations': 200,
  'learning_rate': 0.09237421878009823,
  'depth': 6,
  'l2_leaf_reg': 5,
  'random_strength': 1,
  'bagging_temperature': 0.6091961642353418,
  'border_count': 32,
  'min_data_in_leaf': 26},
 'target_4_1': {'iterations': 100,
  'learning_rate': 0.09210273435223537,
  'depth': 6,
  'l2_leaf_reg': 6,
  'random_strength': 2,
  'bagging_temperature': 0.5683704798044688,
  'border_count': 96,
  'min_data_in_leaf': 14},
 'target_5_1': {'iterations': 200,
  'learning_rate': 0.09237421878009823,
  'depth': 6,
  'l2_leaf_reg': 5,
  'random_strength': 1,
  'bagging_temperature': 0.6091961642353418,
  'border_count': 32,
  'min_data_in_leaf': 26},
 'target_5_2': {'iterations': 100,
  'learning_rate': 0.04437555550097432,
  'depth': 3,
  'l2_leaf_reg': 7,
  'random_strength': 2,
  'bagging_temperature': 0.9637655990477874,
  'border_count': 64,
  'min_data_in_leaf': 16},
 'target_6_1': {'iterations': 200,
  'learning_rate': 0.09237421878009823,
  'depth': 6,
  'l2_leaf_reg': 5,
  'random_strength': 1,
  'bagging_temperature': 0.6091961642353418,
  'border_count': 32,
  'min_data_in_leaf': 26},
 'target_6_2': {'iterations': 200,
  'learning_rate': 0.09237421878009823,
  'depth': 6,
  'l2_leaf_reg': 5,
  'random_strength': 1,
  'bagging_temperature': 0.6091961642353418,
  'border_count': 32,
  'min_data_in_leaf': 26},
 'target_6_3': {'iterations': 250,
  'learning_rate': 0.07076922519051033,
  'depth': 3,
  'l2_leaf_reg': 4,
  'random_strength': 3,
  'bagging_temperature': 0.5325152889039984,
  'border_count': 96,
  'min_data_in_leaf': 6},
 'target_6_4': {'iterations': 250,
  'learning_rate': 0.07076922519051033,
  'depth': 3,
  'l2_leaf_reg': 4,
  'random_strength': 3,
  'bagging_temperature': 0.5325152889039984,
  'border_count': 96,
  'min_data_in_leaf': 6},
 'target_6_5': {'iterations': 100,
  'learning_rate': 0.09210273435223537,
  'depth': 6,
  'l2_leaf_reg': 6,
  'random_strength': 2,
  'bagging_temperature': 0.5683704798044688,
  'border_count': 96,
  'min_data_in_leaf': 14},
 'target_7_1': {'iterations': 200,
  'learning_rate': 0.09237421878009823,
  'depth': 6,
  'l2_leaf_reg': 5,
  'random_strength': 1,
  'bagging_temperature': 0.6091961642353418,
  'border_count': 32,
  'min_data_in_leaf': 26},
 'target_7_2': {'iterations': 200,
  'learning_rate': 0.09237421878009823,
  'depth': 6,
  'l2_leaf_reg': 5,
  'random_strength': 1,
  'bagging_temperature': 0.6091961642353418,
  'border_count': 32,
  'min_data_in_leaf': 26},
 'target_7_3': {'iterations': 250,
  'learning_rate': 0.07076922519051033,
  'depth': 3,
  'l2_leaf_reg': 4,
  'random_strength': 3,
  'bagging_temperature': 0.5325152889039984,
  'border_count': 96,
  'min_data_in_leaf': 6},
 'target_8_1': {'iterations': 200,
  'learning_rate': 0.09237421878009823,
  'depth': 6,
  'l2_leaf_reg': 5,
  'random_strength': 1,
  'bagging_temperature': 0.6091961642353418,
  'border_count': 32,
  'min_data_in_leaf': 26},
 'target_8_2': {'iterations': 200,
  'learning_rate': 0.09237421878009823,
  'depth': 6,
  'l2_leaf_reg': 5,
  'random_strength': 1,
  'bagging_temperature': 0.6091961642353418,
  'border_count': 32,
  'min_data_in_leaf': 26},
 'target_8_3': {'iterations': 200,
  'learning_rate': 0.09237421878009823,
  'depth': 6,
  'l2_leaf_reg': 5,
  'random_strength': 1,
  'bagging_temperature': 0.6091961642353418,
  'border_count': 32,
  'min_data_in_leaf': 26},
 'target_9_1': {'iterations': 250,
  'learning_rate': 0.07076922519051033,
  'depth': 3,
  'l2_leaf_reg': 4,
  'random_strength': 3,
  'bagging_temperature': 0.5325152889039984,
  'border_count': 96,
  'min_data_in_leaf': 6},
 'target_9_2': {'iterations': 200,
  'learning_rate': 0.09237421878009823,
  'depth': 6,
  'l2_leaf_reg': 5,
  'random_strength': 1,
  'bagging_temperature': 0.6091961642353418,
  'border_count': 32,
  'min_data_in_leaf': 26},
 'target_9_3': {'iterations': 200,
  'learning_rate': 0.09237421878009823,
  'depth': 6,
  'l2_leaf_reg': 5,
  'random_strength': 1,
  'bagging_temperature': 0.6091961642353418,
  'border_count': 32,
  'min_data_in_leaf': 26},
 'target_9_4': {'iterations': 250,
  'learning_rate': 0.07076922519051033,
  'depth': 3,
  'l2_leaf_reg': 4,
  'random_strength': 3,
  'bagging_temperature': 0.5325152889039984,
  'border_count': 96,
  'min_data_in_leaf': 6},
 'target_9_5': {'iterations': 300,
  'learning_rate': 0.06251028636335225,
  'depth': 3,
  'l2_leaf_reg': 7,
  'random_strength': 5,
  'bagging_temperature': 0.6486373774747933,
  'border_count': 32,
  'min_data_in_leaf': 6},
 'target_9_6': {'iterations': 200,
  'learning_rate': 0.09237421878009823,
  'depth': 6,
  'l2_leaf_reg': 5,
  'random_strength': 1,
  'bagging_temperature': 0.6091961642353418,
  'border_count': 32,
  'min_data_in_leaf': 26},
 'target_9_7': {'iterations': 200,
  'learning_rate': 0.09237421878009823,
  'depth': 6,
  'l2_leaf_reg': 5,
  'random_strength': 1,
  'bagging_temperature': 0.6091961642353418,
  'border_count': 32,
  'min_data_in_leaf': 26},
 'target_9_8': {'iterations': 300,
  'learning_rate': 0.06251028636335225,
  'depth': 3,
  'l2_leaf_reg': 7,
  'random_strength': 5,
  'bagging_temperature': 0.6486373774747933,
  'border_count': 32,
  'min_data_in_leaf': 6},
 'target_10_1': {'iterations': 200,
  'learning_rate': 0.09237421878009823,
  'depth': 6,
  'l2_leaf_reg': 5,
  'random_strength': 1,
  'bagging_temperature': 0.6091961642353418,
  'border_count': 32,
  'min_data_in_leaf': 26}}

In [39]:
best_params

{'target_1_1': {'iterations': 200,
  'learning_rate': 0.09237421878009823,
  'depth': 6,
  'l2_leaf_reg': 5,
  'random_strength': 1,
  'bagging_temperature': 0.6091961642353418,
  'border_count': 32,
  'min_data_in_leaf': 26},
 'target_1_2': {'iterations': 250,
  'learning_rate': 0.07076922519051033,
  'depth': 3,
  'l2_leaf_reg': 4,
  'random_strength': 3,
  'bagging_temperature': 0.5325152889039984,
  'border_count': 96,
  'min_data_in_leaf': 6},
 'target_1_3': {'iterations': 200,
  'learning_rate': 0.09237421878009823,
  'depth': 6,
  'l2_leaf_reg': 5,
  'random_strength': 1,
  'bagging_temperature': 0.6091961642353418,
  'border_count': 32,
  'min_data_in_leaf': 26},
 'target_1_4': {'iterations': 200,
  'learning_rate': 0.09237421878009823,
  'depth': 6,
  'l2_leaf_reg': 5,
  'random_strength': 1,
  'bagging_temperature': 0.6091961642353418,
  'border_count': 32,
  'min_data_in_leaf': 26},
 'target_1_5': {'iterations': 200,
  'learning_rate': 0.09237421878009823,
  'depth': 6,
  'l

In [32]:
models = {}
val_auc = {}

In [ ]:
def train(target_name):
    print(f"\nОбучаем {target_name}...")
    
    # Загружаем отобранные признаки для этого таргета
    selected_features = load_selected_features(target_name)
    
    if selected_features is None:
        print(f"  ВНИМАНИЕ: Файл с признаками для {target_name} не найден, используем все признаки")
        X_train_selected = X_train
        X_val_selected = X_val
        X_test_selected = X_test
        cat_features_selected = cat_features
    else:
        # Убеждаемся, что все выбранные признаки существуют в данных
        available_features = [f for f in selected_features if f in X_train.columns]
        if len(available_features) < len(selected_features):
            print(f"  Предупреждение: только {len(available_features)} из {len(selected_features)} признаков доступны")
        
        print(f"  Используем {len(available_features)} отобранных признаков")
        # Фильтруем только отобранные признаки
        X_train_selected = X_train[available_features]
        X_val_selected = X_val[available_features]
        X_test_selected = X_test[available_features]
        # Категориальные признаки только из отобранных
        cat_features_selected = [f for f in cat_features if f in available_features]
    
    # Копируем параметры и добавляем обязательные
    if target_name in best_params:
        params = best_params[target_name].copy()
    else:
        # Дефолтные параметры, если нет в best_params
        params = {
            'iterations': 300,
            'learning_rate': 0.06,
            'depth': 5,
            'l2_leaf_reg': 4,
            'random_strength': 2,
            'bagging_temperature': 0.7,
            'border_count': 128,
            'min_data_in_leaf': 10
        }
    
    # Увеличиваем iterations для финальной модели
    if 'iterations' in params:
        params['iterations'] = int(params['iterations'] * 1.2)
    
    # Удаляем early_stopping_rounds (будем использовать раннюю остановку отдельно)
    params.pop('early_stopping_rounds', None)
    
    # Добавляем обязательные параметры для CatBoost
    params['eval_metric'] = 'AUC'
    params['loss_function'] = 'Logloss'
    params['random_seed'] = SEED
    params['verbose'] = False
    params['task_type']="GPU"
    params['devices']='0'
    params['metric_period']=50
    
    # Добавляем веса классов для борьбы с дисбалансом
    pos = y_train[target_name].sum()
    neg = len(y_train) - pos
    if pos > 0 and neg > 0:
        params['class_weights'] = {0: 1, 1: neg / pos}
    
    # Выводим параметры для информации
    print(f"  Параметры: iterations={params.get('iterations', 'N/A')}, "
          f"depth={params.get('depth', 'N/A')}, "
          f"lr={params.get('learning_rate', 'N/A'):.3f}")
    
    # Создаем модель
    model = CatBoostClassifier(**params)
    
    # Создаем Pool
    train_pool = Pool(X_train_selected, label=y_train[target_name], cat_features=cat_features_selected)
    val_pool = Pool(X_val_selected, label=y_val[target_name], cat_features=cat_features_selected)
    
    # Обучаем с ранней остановкой
    model.fit(
        train_pool,
        eval_set=val_pool,
        early_stopping_rounds=50,
        verbose=False
    )
    
    # Оценка на валидации
    pred_val = model.predict_proba(val_pool)[:, 1]
    auc = roc_auc_score(y_val[target_name], pred_val)
    print(f"  ROC-AUC: {auc:.4f}")
    models[target_name]=model

In [37]:
need_selection = False
for target_name in target_cols[:5]:  # Проверяем первые 5 для примера
    if not os.path.exists(f"{SELECTED_FEATURES_PATH}/{target_name}.txt"):
        need_selection = True
        break
    
if need_selection:
    print("\n⚠️  Файлы с отобранными признаками не найдены. Запускаем отбор...")
    select_features_for_all_targets()
else:
    print("\n✓ Файлы с отобранными признаками найдены. Переходим к обучению.")


⚠️  Файлы с отобранными признаками не найдены. Запускаем отбор...

ЭТАП 1: ОТБОР ПРИЗНАКОВ ДЛЯ КАЖДОГО ТАРГЕТА

----- Отбор признаков для target_1_1 -----
  Положительных: 7797, всего: 750000
  Размер подвыборки: 170000
  Загружено признаков: 2440
  Выбрано 350 признаков
  ✓ Сохранено в data/selected_features/target_1_1.txt

----- Отбор признаков для target_1_2 -----
  Положительных: 2569, всего: 750000
  Размер подвыборки: 170000
  Загружено признаков: 2440
  Выбрано 350 признаков
  ✓ Сохранено в data/selected_features/target_1_2.txt

----- Отбор признаков для target_1_3 -----
  Положительных: 17839, всего: 750000
  Размер подвыборки: 170000
  Загружено признаков: 2440
  Выбрано 350 признаков
  ✓ Сохранено в data/selected_features/target_1_3.txt

----- Отбор признаков для target_1_4 -----
  Положительных: 17572, всего: 750000
  Размер подвыборки: 170000
  Загружено признаков: 2440
  Выбрано 350 признаков
  ✓ Сохранено в data/selected_features/target_1_4.txt

----- Отбор признаков для

In [42]:
# group 1
for target_name in groups[0]:
    train(target_name)


Обучаем target_1_1...
  Используем 350 отобранных признаков
  Параметры: iterations=240, depth=6, lr=0.092
  ROC-AUC: 0.9153

Обучаем target_1_2...
  Используем 350 отобранных признаков
  Параметры: iterations=300, depth=3, lr=0.071
  ROC-AUC: 0.8278

Обучаем target_1_3...
  Используем 350 отобранных признаков
  Параметры: iterations=240, depth=6, lr=0.092
  ROC-AUC: 0.8689

Обучаем target_1_4...
  Используем 350 отобранных признаков
  Параметры: iterations=240, depth=6, lr=0.092
  ROC-AUC: 0.8364

Обучаем target_1_5...
  Используем 350 отобранных признаков
  Параметры: iterations=240, depth=6, lr=0.092
  ROC-AUC: 0.8829

Обучаем target_2_1...
  Используем 350 отобранных признаков
  Параметры: iterations=240, depth=6, lr=0.092
  ROC-AUC: 0.8339

Обучаем target_2_2...
  Используем 350 отобранных признаков
  Параметры: iterations=240, depth=6, lr=0.092
  ROC-AUC: 0.9363

Обучаем target_2_3...
  Используем 350 отобранных признаков
  Параметры: iterations=240, depth=6, lr=0.092
  ROC-AUC:

In [44]:
# group 2
for target_name in groups[1]:
    train(target_name)


Обучаем target_2_6...
  Используем 350 отобранных признаков
  Параметры: iterations=300, depth=3, lr=0.071
  ROC-AUC: 0.7464

Обучаем target_2_7...
  Используем 350 отобранных признаков
  Параметры: iterations=120, depth=6, lr=0.092
  ROC-AUC: 0.8383

Обучаем target_2_8...
  Используем 350 отобранных признаков
  Параметры: iterations=240, depth=6, lr=0.092
  ROC-AUC: 0.9949

Обучаем target_3_1...
  Используем 350 отобранных признаков
  Параметры: iterations=240, depth=6, lr=0.092
  ROC-AUC: 0.6939

Обучаем target_3_2...
  Используем 350 отобранных признаков
  Параметры: iterations=240, depth=6, lr=0.092
  ROC-AUC: 0.9112

Обучаем target_3_3...
  Используем 350 отобранных признаков
  Параметры: iterations=240, depth=6, lr=0.092
  ROC-AUC: 0.7296

Обучаем target_3_4...
  Используем 350 отобранных признаков
  Параметры: iterations=300, depth=3, lr=0.071
  ROC-AUC: 0.9437

Обучаем target_3_5...
  Используем 350 отобранных признаков
  Параметры: iterations=240, depth=6, lr=0.092
  ROC-AUC:

In [46]:
# group 3
for target_name in groups[2]:
    train(target_name)


Обучаем target_5_2...
  Используем 350 отобранных признаков
  Параметры: iterations=120, depth=3, lr=0.044
  ROC-AUC: 0.7433

Обучаем target_6_1...
  Используем 350 отобранных признаков
  Параметры: iterations=240, depth=6, lr=0.092
  ROC-AUC: 0.7211

Обучаем target_6_2...
  Используем 350 отобранных признаков
  Параметры: iterations=240, depth=6, lr=0.092
  ROC-AUC: 0.7275

Обучаем target_6_3...
  Используем 350 отобранных признаков
  Параметры: iterations=300, depth=3, lr=0.071
  ROC-AUC: 0.7719

Обучаем target_6_4...
  Используем 350 отобранных признаков
  Параметры: iterations=300, depth=3, lr=0.071
  ROC-AUC: 0.8512

Обучаем target_6_5...
  Используем 350 отобранных признаков
  Параметры: iterations=120, depth=6, lr=0.092
  ROC-AUC: 0.9340

Обучаем target_7_1...
  Используем 350 отобранных признаков
  Параметры: iterations=240, depth=6, lr=0.092
  ROC-AUC: 0.8074

Обучаем target_7_2...
  Используем 350 отобранных признаков
  Параметры: iterations=240, depth=6, lr=0.092
  ROC-AUC:

In [48]:
# group 4
for target_name in groups[3]:
    train(target_name)


Обучаем target_8_2...
  Используем 350 отобранных признаков
  Параметры: iterations=240, depth=6, lr=0.092
  ROC-AUC: 0.8503

Обучаем target_8_3...
  Используем 350 отобранных признаков
  Параметры: iterations=240, depth=6, lr=0.092
  ROC-AUC: 0.8807

Обучаем target_9_1...
  Используем 350 отобранных признаков
  Параметры: iterations=300, depth=3, lr=0.071
  ROC-AUC: 0.7789

Обучаем target_9_2...
  Используем 350 отобранных признаков
  Параметры: iterations=240, depth=6, lr=0.092
  ROC-AUC: 0.8345

Обучаем target_9_3...
  Используем 350 отобранных признаков
  Параметры: iterations=240, depth=6, lr=0.092
  ROC-AUC: 0.6812

Обучаем target_9_4...
  Используем 350 отобранных признаков
  Параметры: iterations=300, depth=3, lr=0.071
  ROC-AUC: 0.9067

Обучаем target_9_5...
  Используем 350 отобранных признаков
  Параметры: iterations=360, depth=3, lr=0.063
  ROC-AUC: 0.8516

Обучаем target_9_6...
  Используем 350 отобранных признаков
  Параметры: iterations=240, depth=6, lr=0.092
  ROC-AUC:

In [54]:
# ------------------------------
# 5. Предсказание на тесте (оптимизированная версия)
# ------------------------------
print("\nФормирование предсказаний...")

# Загружаем тестовые данные один раз
X_test_full = test_wide.to_pandas()
print(f"Загружено тестовых данных: {X_test_full.shape}")

# Определяем все категориальные признаки
all_cat_features = cat_features

# Создаём словарь для кэширования обработанных данных
test_data_cache = {}

# Подготавливаем данные для каждого таргета (но не для предсказания)
print("Подготовка данных для каждого таргета...")
for target_name in target_cols:
    # Загружаем отобранные признаки
    selected_features = load_selected_features(target_name)
    
    if selected_features is None or len(selected_features) == 0:
        # Если нет отобранных признаков, используем все
        available_features = X_test_full.columns.tolist()
        cat_features_selected = all_cat_features
    else:
        # Фильтруем доступные признаки
        available_features = [f for f in selected_features if f in X_test_full.columns]
        cat_features_selected = [f for f in all_cat_features if f in available_features]
    
    # Создаём копию данных только с нужными признаками
    X_test_selected = X_test_full[available_features].copy()
    
    # Преобразуем категориальные признаки (один раз для каждого таргета)
    for col in cat_features_selected:
        if col in X_test_selected.columns:
            X_test_selected[col] = X_test_selected[col].fillna(-1).astype(int)
    
    # Сохраняем в кэш
    test_data_cache[target_name] = (X_test_selected, cat_features_selected)

print(f"Данные подготовлены для {len(test_data_cache)} таргетов")

# Теперь делаем предсказания (быстро, так как данные уже подготовлены)
print("\nВыполнение предсказаний...")
predictions = []

for i, target_name in enumerate(target_cols):
    print(f"  {i+1}/{len(target_cols)}: {target_name}...", end=" ", flush=True)
    
    # Получаем подготовленные данные из кэша
    X_test_selected, cat_features_selected = test_data_cache[target_name]
    
    # Получаем модель
    model = models[target_name]
    
    # Создаём Pool и предсказываем
    test_pool = Pool(X_test_selected, cat_features=cat_features_selected)
    proba = model.predict_proba(test_pool)[:, 1]
    predictions.append(proba)
    
    print("готово")

# Формируем результат
predictions = np.column_stack(predictions)
predict_cols = [f"predict_{col.replace('target_', '')}" for col in target_cols]
pred_df = pl.DataFrame(predictions, schema=predict_cols)

# Сохраняем сабмит
submit = pl.DataFrame({'customer_id': test_customer_ids.to_pandas()['customer_id']}).hstack(pred_df)
submit.write_parquet("data/sample_submit_final.parquet")

print("\n✅ Сабмит сохранён в data/sample_submit_final.parquet")
print(f"   Размер: {submit.shape[0]} строк, {submit.shape[1]} колонок")


Формирование предсказаний...
Загружено тестовых данных: (250000, 2440)
Подготовка данных для каждого таргета...
Данные подготовлены для 41 таргетов

Выполнение предсказаний...
  1/41: target_1_1... готово
  2/41: target_1_2... готово
  3/41: target_1_3... готово
  4/41: target_1_4... готово
  5/41: target_1_5... готово
  6/41: target_2_1... готово
  7/41: target_2_2... готово
  8/41: target_2_3... готово
  9/41: target_2_4... готово
  10/41: target_2_5... готово
  11/41: target_2_6... готово
  12/41: target_2_7... готово
  13/41: target_2_8... готово
  14/41: target_3_1... готово
  15/41: target_3_2... готово
  16/41: target_3_3... готово
  17/41: target_3_4... готово
  18/41: target_3_5... готово
  19/41: target_4_1... готово
  20/41: target_5_1... готово
  21/41: target_5_2... готово
  22/41: target_6_1... готово
  23/41: target_6_2... готово
  24/41: target_6_3... готово
  25/41: target_6_4... готово
  26/41: target_6_5... готово
  27/41: target_7_1... готово
  28/41: target_7_2...